# NullFusion v4 -- Multi-scale Spectral Dictionary + Wavelet High-Freq Branch
Novel: Pyramid dictionary (global+mid+fine) + Wavelet high-freq branch on null-space residual + Spectral consistency.
Target: beat FeINFN 52.47 dB on CAVE x4 (Nikon D700, Wald blur).

In [ ]:
import subprocess, os, shutil

result = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(f'GPU: {result.stdout.strip()}')
if '6.0' in result.stdout or 'P100' in result.stdout:
    print('P100 detected (sm_60), need torch 2.5.1+cu121...')
    whls = []
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f.endswith('.whl'):
                whls.append(os.path.join(root, f))
    whls.sort()
    print(f'Found wheels: {whls}')
    for w in whls:
        base = os.path.basename(w)
        fixed = base.replace('cu121-cp312', '+cu121-cp312')
        dst = os.path.join('/tmp', fixed)
        shutil.copy2(w, dst)
        print(f'Installing {dst}')
        subprocess.check_call(['pip', 'install', '--force-reinstall', '--no-deps', dst])
    print('torch install done')

In [ ]:
import torch
print(f'torch {torch.__version__}')
assert torch.cuda.is_available(), 'CUDA not available'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name} sm_{p.major}{p.minor}')

In [ ]:
!pip install -q scipy scikit-image

In [ ]:
import os, sys, shutil

SCRIPT = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == 'train_nullfusion_v3.py':
            SCRIPT = os.path.join(root, f)
            break
    if SCRIPT:
        break
print(f'Script: {SCRIPT}')

shutil.copy(SCRIPT, '/kaggle/working/train_nullfusion_v3.py')
print('Copied to /kaggle/working/')
os.chdir('/kaggle/working')
print(os.getcwd())

In [ ]:
!python train_nullfusion_v3.py --root /kaggle/input/datasets/liptee/hyperspectral-image-restoration-based-on-cave --output_dir /kaggle/working